# 1. Load data

This notebook produces results for the RF, ET, XGB, and LGBM results. Tune period and excess returns in this cell. Results are output in `tree_runs.csv`, which we manually clean up for clearer reporting.

In [7]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import seaborn as sns
import utils.base_utils as bu
import utils.window_utils as wu
import numpy as np
import matplotlib.pyplot as plt
from utils.publication_lags import apply_fred_md_publication_lag
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array

repo_root = os.path.abspath('../..')

# Bianchi period:
start_date = '1971-08-31'
end_date = '2018-12-31'
# end_date = '2025-06-30' # kr and gsw end date
DATASET = 'lw' # or 'kr
GAP = 0
REVISED = True # Tune manually, True if use FRED, False if use ALFRED.

maturities = [str(i) for i in range(12, 121) if i % 12 == 0] # select only yearly maturities

yields = bu.get_yields(type=DATASET, start=start_date, end=end_date, maturities=maturities) # type can be kr, lw, gsw
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna() # horizon=12 means holding for 12 months

# adjust fred_md start_date by 6 months to fetch enough data for shifting
fred_md_start_date = pd.to_datetime(start_date) - pd.DateOffset(months=6)
fred_md_raw = bu.get_fred_data('data/2026-01-MD.csv', start=fred_md_start_date, end=end_date) # this is aligned to the last day of the previous month, so we get the same number of observations as the yields data

monthly_yields = bu.get_yields(type=DATASET, start=start_date, end=end_date, maturities=[str(i) for i in range(1, 121)]) # needed for monthly holding period excess returns. Not available for gsw
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna() # calculate monthly excess returns for robustness

# If wanted, apply per-series publication lag to latest-snapshot macro data
# from utils.publication_lags import apply_fred_md_publication_lag
# fred_md = apply_fred_md_publication_lag(fred_md_raw)  
# For results in paper, we naively shift all FRED-MD series by 1 month
# to reflect publication lag:
fred_md = fred_md_raw.shift(1)

fred_md_realtime = pd.read_csv(
    os.path.join(repo_root, 'data', 'ALFRED', 'simple_outputs', 'realtime_final_balanced.csv'),
    parse_dates=True,
    index_col=0
 )
fred_md_realtime.index.name = 'date'

# Drop TWEXAFEGSMTHx and ACOGNO as they start late
fred_md = fred_md.drop(columns=['TWEXAFEGSMTHx', 'ACOGNO'])
fred_md_realtime = fred_md_realtime.drop(columns=['TWEXAFEGSMTHx', 'ACOGNO'], errors='ignore')
# Finally, revert fred_md to start_date, after transformations and lag adjustments
fred_md = fred_md[start_date:end_date]
fred_md_realtime = fred_md_realtime[start_date:end_date]

# Drop dates outside the xr range
yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]
fred_md_realtime = fred_md_realtime.loc[fred_md_realtime.index <= xr.index[-1]].reindex(xr.index)
monthly_xr = monthly_xr.loc[monthly_xr.index <= xr.index[-1]]

# Construct X with 3-level MultiIndex: (source, group, series)
s2g = build_full_group_mapping(fred_md, forward, yields)

X = pd.concat([fred_md, forward, yields],
               axis=1,
               keys=['fred', 'forward', 'yields'])

X = add_group_level(X, s2g, level_name='group')
X = X.sort_index(axis=1, level='group')
groups = groups_as_array(X, level='group')

X_realtime = pd.concat([fred_md_realtime, forward, yields],
                        axis=1,
                        keys=['fred', 'forward', 'yields'])
X_realtime = add_group_level(X_realtime, s2g, level_name='group')
X_realtime = X_realtime.sort_index(axis=1, level='group')
groups_realtime = groups_as_array(X_realtime, level='group')

y_all = xr[['24','36','48','60','72','84','96','108','120']].values
dates = xr.index

/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:219: UserWarning: The following series are defined in get_fredmd_grouping() but are not present in the FRED-MD data: ['ACOGNO', 'TWEXAFEGSMTHx']. They may have been dropped or renamed.
  warnings.warn(
/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:168: UserWarning: 2 entries in series_to_group are not present in the DataFrame columns: ['ACOGNO', 'TWEXAFEGSMTHx']
  warnings.warn(
/home/ulrikts/Documents/NTNU/TIO4900-Replication/utils/macro_grouping.py:168: UserWarning: 2 entries in series_to_group are not present in the DataFrame columns: ['ACOGNO', 'TWEXAFEGSMTHx']
  warnings.warn(


# 2. Run models

In [ ]:
from models.base import *
from models.classical import *
from models.gbt import *
from models.linear import *
from models.tree import *
from tqdm import tqdm
from joblib import Parallel, delayed
from datetime import datetime

# y = monthly_xr['120'].values # 1-month excess returns
# y = xr['120'].values # 10-year overlapping excess returns
OOS_start = pd.Timestamp('1990-01-31')

coefs = []

def record_coef(coef):
    coefs.append(np.abs(coef).copy())  # store absolute value

tree_models = {
    # "Tree_RF": FwdFredTreeEnsemble1D(estimator="rf"),
    # "Tree_ET": FwdFredTreeEnsemble1D(estimator="ef"),
    "Tree_XGB": FwdFredTreeEnsemble1D(estimator="xgb"),
    "Tree_LGBM": FwdFredTreeEnsemble1D(estimator="lgbm"),

    # # Forward-only variants:
    # "Tree_RF_FwdOnly": FwdFredTreeEnsemble1D(estimator="rf", include_fred=False),
    # "Tree_ET_FwdOnly": FwdFredTreeEnsemble1D(estimator="ef", include_fred=False),
    # "Tree_XGB_FwdOnly": FwdFredTreeEnsemble1D(estimator="xgb", include_fred=False),
    # "FwdOnly": FwdFredTreeEnsemble1D(estimator="lgbm", include_fred=False)
}

all_models = {**tree_models}

results = {}

# Helper function for parallel evaluation of a single model+maturity combination
def evaluate_model_maturity(name, model, maturity, X, y_dict, dates, OOS_start):
    """
    Evaluate a single model on a single maturity in isolation.
    Returns a tuple: (model_name, maturity, r2, significance)
    """
    try:
        y = y_dict[maturity]
        y_forecast = wu.expanding_window(
            model, X, y, dates, OOS_start, 
            gap=0,         # gap = 11 for annual nonoverlapping yields
            refit_freq=1,  # Refit every month
        )
        r2 = wu.oos_r2(y, y_forecast)
        signif = bu.RSZ_Signif(y, y_forecast)
        return (name, maturity, r2, signif)
    except Exception as e:
        print(f"Error evaluating {name} on maturity {maturity}: {e}")
        return (name, maturity, np.nan, np.nan)

# Prepare y_dict for parallel execution
y_dict = {mat: xr[mat].values for mat in ['24','36','48','60','84','120']}

# Create list of (model_name, model_instance, maturity) tuples
task_list = [
    (name, model, maturity)
    for name, model in all_models.items()
    for maturity in ['24','36','48','60','84','120']
    # for maturity in ['120']
]

print(f"Running {len(task_list)} model-maturity combinations in parallel...")

# Run all combinations in parallel using joblib
# n_jobs=-1 uses all available CPU cores
results_list = Parallel(n_jobs=-1, verbose=1)(
    delayed(evaluate_model_maturity)(name, model, maturity, X, y_dict, dates, OOS_start)
    for name, model, maturity in task_list
)

# Display results
print("\n=== RESULTS ===")
for name, maturity, r2, signif in results_list:
    print(f"Model: {name}\tMaturity {maturity}:\n R2_OOS = {r2:.4f}\n Significance: {signif:.3f}\n")

# Save results to tree_results.csv
csv_path = os.path.join(repo_root, 'notebooks', 'orchestrators', 'results', 'tree_results.csv')

# Convert results to DataFrame with proper columns
results_df_list = []
for name, maturity, r2, signif in results_list:
    results_df_list.append({
        'time_now': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'name': name,
        'maturity': maturity,
        'signif': signif,
        'r2': r2,
        'start_date': start_date,
        'end_date': end_date,
        'yield_type': f'{DATASET}',
        'f"Revised macro data: {revised}"': f'Revised macro data: {REVISED}',
        'f"gap={gap}"': f'gap={GAP}'
    })

results_df = pd.DataFrame(results_df_list)

# Append to existing CSV or create new one
if os.path.exists(csv_path):
    existing_df = pd.read_csv(csv_path)
    # Strip whitespaces from column names to avoid duplication
    existing_df.columns = existing_df.columns.str.strip()
    # Drop duplicated columns to prevent InvalidIndexError during concat
    existing_df = existing_df.loc[:, ~existing_df.columns.duplicated()]
    results_df = pd.concat([existing_df, results_df], ignore_index=True)

results_df.to_csv(csv_path, index=False)
print(f"\nResults saved to {csv_path}")

Running 1 model-maturity combinations in parallel...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.



=== RESULTS ===
Model: FwdOnly	Maturity 120:
 R2_OOS = 0.1554
 Significance: 0.000



InvalidIndexError: Reindexing only valid with uniquely valued Index objects